# Remaining SLM MCQ Selective Prediction Experiments: Gemma + Llama 3.1 + Stronger Hidden-State Probe

This notebook is **fully inline**. It does **not** call any external `.py` file.

It runs the remaining reviewer-strengthening experiments for:

- `google/gemma-2-9b-it`
- `meta-llama/Llama-3.1-8B-Instruct`

It includes:

- confidence/option-score features;
- hidden-state summary features;
- PCA-compressed hidden embeddings;
- **supervised contrastive hidden-state probe**;
- selective-prediction metrics: Risk@80/60/40 coverage, AURC, E-AURC, failure capture at 20% abstention;
- PCA-dimension sensitivity;
- layer-localization analysis;
- group-permutation/redundancy analysis;
- latency/VRAM diagnostics;
- automatic CSV tables, figures, and ZIP export.

**Kaggle settings:** GPU on, Internet on, `HF_TOKEN` added in Add-ons → Secrets and enabled for this notebook.

Run the notebook in debug mode first, then restart the session and run full mode.

In [ ]:
# Install dependencies. Run this once at the beginning of the Kaggle session.
!pip -q install -U transformers accelerate bitsandbytes datasets scikit-learn scipy pandas numpy matplotlib pyarrow huggingface_hub tqdm

## 1. Configuration

For a quick test, keep `DEBUG_N = 20`. After it succeeds, restart the Kaggle session and set `DEBUG_N = None` for the full run.

The full run uses up to 1000 examples for CommonsenseQA, HellaSwag, and MMLU; ARC-Challenge uses its validation size.

In [ ]:
import os
from pathlib import Path

# =========================
# USER SETTINGS
# =========================
DEBUG_N = None        # paper reproduction mode; set to 20 for a quick debug run
OVERWRITE = False     # keep False for resumable runs; True forces recomputation
RUN_FEATURE_EXTRACTION = True
RUN_ANALYSIS = True
RUN_LATENCY = True
RUN_REPEATED_SAMPLING_FEATURES = False  # expensive; latency benchmark still measures 5x generation

# Models requested by the user.
MODEL_SPECS = [
    {
        "short_name": "gemma2_9b_it",
        "model_id": "google/gemma-2-9b-it",
        "family": "Gemma-2-9B-IT",
    },
    {
        "short_name": "llama31_8b_instruct",
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "family": "Llama-3.1-8B-Instruct",
    },
]

DATASET_NAMES = ["arc_challenge", "commonsenseqa", "hellaswag", "mmlu"]

# Full-run sample sizes. DEBUG_N overrides these.
MAX_EXAMPLES = {
    "arc_challenge": 299,
    "commonsenseqa": 1000,
    "hellaswag": 1000,
    "mmlu": 1000,
}

# Feature/probe settings
MAX_OPTIONS = 5
MAX_PROMPT_TOKENS = 1024
PCA_MAIN_DIM = 64
PCA_DIMS = [8, 16, 32, 64, 128, 256]
N_SPLITS = 5
TEST_SIZE = 0.30
RANDOM_SEED = 42
CONTRASTIVE_EPOCHS = 35 if DEBUG_N is None else 5
CONTRASTIVE_PRE_PCA_DIM = 128 if DEBUG_N is None else 32
CONTRASTIVE_EMB_DIM = 32
CONTRASTIVE_BATCH_SIZE = 128

# Output folder
OUT_DIR = Path("/kaggle/working/slm_gemma_llama_inline_outputs") if Path("/kaggle").exists() else Path("./slm_gemma_llama_inline_outputs")
FEATURE_DIR = OUT_DIR / "features"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
LOG_DIR = OUT_DIR / "logs"
for d in [OUT_DIR, FEATURE_DIR, TABLE_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)
print("DEBUG_N:", DEBUG_N)
print("Models:", [m["model_id"] for m in MODEL_SPECS])

In [ ]:
# Imports and deterministic setup
import json
import math
import time
import gc
import random
import zipfile
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.special import logsumexp
from scipy.stats import bootstrap

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score,
)
from sklearn.inspection import permutation_importance

from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

warnings.filterwarnings("ignore")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
# NumPy compatibility: some recent Kaggle/NumPy builds expose np.trapezoid but not np.trapz.
# The selective-prediction AURC/E-AURC code uses np.trapz for backward compatibility.
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid
print("NumPy version:", np.__version__, "| np.trapz available:", hasattr(np, "trapz"))


## 2. Hugging Face login

This cell reads your Kaggle secret named `HF_TOKEN`. You must have accepted the licenses/access pages for Gemma and Llama on Hugging Face before running.

In [ ]:
def get_hf_token():
    token = os.environ.get("HF_TOKEN", None)
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        return token
    except Exception as e:
        print("Could not read Kaggle secret HF_TOKEN:", repr(e))
        return None

HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Add it in Kaggle Add-ons → Secrets with the exact name HF_TOKEN.")

login(token=HF_TOKEN)
print("Logged in to Hugging Face. Token is available, but not printed for security.")

## 3. Dataset loading and prompt formatting

In [ ]:
LABELS = ["A", "B", "C", "D", "E"]


def safe_str(x):
    if x is None:
        return ""
    return str(x).replace("\n", " ").strip()


def answer_key_to_index(answer, labels=None):
    """Convert answer keys such as 'A', 'B', 0, 1, '1' to option index."""
    if labels is None:
        labels = LABELS
    if isinstance(answer, (int, np.integer)):
        return int(answer)
    s = str(answer).strip()
    if s in labels:
        return labels.index(s)
    if s.upper() in labels:
        return labels.index(s.upper())
    if s.isdigit():
        val = int(s)
        if 0 <= val < len(labels):
            return val
        if 1 <= val <= len(labels):
            return val - 1
    return None


def load_arc_challenge(max_n):
    ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="validation")
    rows = []
    for i, ex in enumerate(ds):
        labels = list(ex["choices"]["label"])
        texts = [safe_str(t) for t in ex["choices"]["text"]]
        gold = answer_key_to_index(ex["answerKey"], labels=labels)
        if gold is None or gold >= len(texts):
            continue
        rows.append({
            "dataset": "arc_challenge",
            "id": f"arc_challenge_{i}",
            "question": safe_str(ex["question"]),
            "choices": texts,
            "labels": labels,
            "answer_idx": gold,
        })
        if max_n is not None and len(rows) >= max_n:
            break
    return rows


def load_commonsenseqa(max_n):
    try:
        ds = load_dataset("tau/commonsense_qa", split="validation")
    except Exception:
        ds = load_dataset("commonsense_qa", split="validation")
    rows = []
    for i, ex in enumerate(ds):
        labels = list(ex["choices"]["label"])
        texts = [safe_str(t) for t in ex["choices"]["text"]]
        gold = answer_key_to_index(ex["answerKey"], labels=labels)
        if gold is None or gold >= len(texts):
            continue
        rows.append({
            "dataset": "commonsenseqa",
            "id": f"commonsenseqa_{i}",
            "question": safe_str(ex["question"]),
            "choices": texts,
            "labels": labels,
            "answer_idx": gold,
        })
        if max_n is not None and len(rows) >= max_n:
            break
    return rows


def load_hellaswag(max_n):
    ds = load_dataset("Rowan/hellaswag", split="validation")
    rows = []
    for i, ex in enumerate(ds):
        ctx = safe_str(ex.get("ctx", ""))
        activity = safe_str(ex.get("activity_label", ""))
        question = f"Choose the most plausible ending. Activity: {activity}. Context: {ctx}"
        choices = [safe_str(t) for t in ex["endings"]]
        gold = answer_key_to_index(ex["label"], labels=LABELS[:len(choices)])
        if gold is None or gold >= len(choices):
            continue
        rows.append({
            "dataset": "hellaswag",
            "id": f"hellaswag_{i}",
            "question": question,
            "choices": choices,
            "labels": LABELS[:len(choices)],
            "answer_idx": gold,
        })
        if max_n is not None and len(rows) >= max_n:
            break
    return rows


def load_mmlu(max_n):
    last_error = None
    candidates = [
        ("cais/mmlu", "all", "validation"),
        ("cais/mmlu", "all", "test"),
        ("lukaemon/mmlu", None, "validation"),
    ]
    ds = None
    for path, config, split in candidates:
        try:
            if config is None:
                ds = load_dataset(path, split=split)
            else:
                ds = load_dataset(path, config, split=split)
            print(f"Loaded MMLU from {path}/{config}/{split}")
            break
        except Exception as e:
            last_error = e
            ds = None
    if ds is None:
        raise RuntimeError(f"Could not load MMLU. Last error: {last_error}")
    rows = []
    for i, ex in enumerate(ds):
        if "choices" in ex:
            choices = [safe_str(t) for t in ex["choices"]]
        elif all(k in ex for k in ["A", "B", "C", "D"]):
            choices = [safe_str(ex[k]) for k in ["A", "B", "C", "D"]]
        else:
            continue
        q = safe_str(ex.get("question", ex.get("input", "")))
        ans = ex.get("answer", ex.get("target", None))
        gold = answer_key_to_index(ans, labels=LABELS[:len(choices)])
        if gold is None or gold >= len(choices):
            continue
        rows.append({
            "dataset": "mmlu",
            "id": f"mmlu_{i}",
            "question": q,
            "choices": choices,
            "labels": LABELS[:len(choices)],
            "answer_idx": gold,
        })
        if max_n is not None and len(rows) >= max_n:
            break
    return rows


def load_dataset_rows(dataset_name, max_n):
    if dataset_name == "arc_challenge":
        return load_arc_challenge(max_n)
    if dataset_name == "commonsenseqa":
        return load_commonsenseqa(max_n)
    if dataset_name == "hellaswag":
        return load_hellaswag(max_n)
    if dataset_name == "mmlu":
        return load_mmlu(max_n)
    raise ValueError(dataset_name)


def format_mcq_prompt(example):
    lines = []
    lines.append("You are answering a multiple-choice question.")
    lines.append("Select the single best answer. Reply with only the option letter.")
    lines.append("")
    lines.append(f"Question: {example['question']}")
    for lab, text in zip(example["labels"], example["choices"]):
        lines.append(f"{lab}. {text}")
    lines.append("")
    lines.append("Answer:")
    return "\n".join(lines)


def apply_chat_template_if_available(tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    if getattr(tokenizer, "chat_template", None):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

# Quick dataset sanity check
for name in DATASET_NAMES:
    n = DEBUG_N if DEBUG_N is not None else min(5, MAX_EXAMPLES[name])
    rows = load_dataset_rows(name, n)
    print(name, len(rows), rows[0]["question"][:80], rows[0]["labels"], rows[0]["answer_idx"])

## 4. Model loading and feature extraction helpers

In [ ]:
def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def load_quantized_model_and_tokenizer(model_id):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=HF_TOKEN,
        use_fast=True,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        attn_implementation="eager",  # safest for hidden-state extraction on Kaggle
    )
    model.eval()
    return tokenizer, model


def get_candidate_token_ids(tokenizer, labels):
    """Collect likely single-token IDs for A/B/C/D/E style next-token scoring."""
    out = {}
    variants_template = ["{}", " {}", "\n{}", "\n {}", "({})", " {}."]
    vocab_size = len(tokenizer)
    for lab in labels:
        ids = set()
        for temp in variants_template:
            s = temp.format(lab)
            toks = tokenizer.encode(s, add_special_tokens=False)
            if len(toks) == 1:
                ids.add(int(toks[0]))
        # fallback: use the final token of label encoding
        if not ids:
            toks = tokenizer.encode(lab, add_special_tokens=False)
            if toks:
                ids.add(int(toks[-1]))
        ids = sorted([i for i in ids if 0 <= i < vocab_size])
        out[lab] = ids
    return out


def entropy_from_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-12, 1.0)
    return float(-(p * np.log(p)).sum())


def summarize_vec(v):
    v = np.asarray(v, dtype=np.float32)
    return [
        float(v.mean()),
        float(v.std()),
        float(np.linalg.norm(v)),
        float(v.min()),
        float(v.max()),
        float(np.mean(np.abs(v))),
    ]


def selected_hidden_indices(hidden_states):
    """Return layer indexes for early/middle/late/final from hidden_states tuple."""
    max_idx = len(hidden_states) - 1
    early = max(1, int(round(0.25 * max_idx)))
    middle = max(1, int(round(0.50 * max_idx)))
    late = max(1, int(round(0.75 * max_idx)))
    final = max_idx
    return {"early": early, "middle": middle, "late": late, "final": final}


def score_one_example(example, tokenizer, model):
    prompt = format_mcq_prompt(example)
    prompt = apply_chat_template_if_available(tokenizer, prompt)
    labels = example["labels"]
    cand_ids = get_candidate_token_ids(tokenizer, labels)

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
        logits = out.logits[0, -1, :].float()
        log_probs_full = torch.log_softmax(logits, dim=-1).detach().cpu().numpy()
        hidden_states = out.hidden_states

    # Label log-probabilities from next-token distribution.
    label_logps = []
    for lab in labels:
        ids = cand_ids.get(lab, [])
        if not ids:
            label_logps.append(-1e9)
        else:
            label_logps.append(float(logsumexp(log_probs_full[ids])))
    label_logps = np.asarray(label_logps, dtype=np.float64)
    probs = np.exp(label_logps - logsumexp(label_logps))
    pred_idx = int(np.argmax(probs))
    correct = int(pred_idx == int(example["answer_idx"]))

    # Confidence features.
    sorted_probs = np.sort(probs)[::-1]
    top_prob = float(sorted_probs[0])
    second_prob = float(sorted_probs[1]) if len(sorted_probs) > 1 else 0.0
    margin = float(top_prob - second_prob)
    ent = entropy_from_probs(probs)
    logit_margin = float(np.sort(label_logps)[-1] - np.sort(label_logps)[-2]) if len(label_logps) > 1 else 0.0

    cheap = [
        top_prob,
        second_prob,
        margin,
        ent,
        logit_margin,
        float(np.max(label_logps)),
        float(np.mean(label_logps)),
        float(np.std(label_logps)),
    ]
    padded_probs = list(probs[:MAX_OPTIONS]) + [0.0] * max(0, MAX_OPTIONS - len(probs))
    padded_logps = list(label_logps[:MAX_OPTIONS]) + [-1e9] * max(0, MAX_OPTIONS - len(label_logps))
    cheap.extend([float(x) for x in padded_probs])
    cheap.extend([float(x) for x in padded_logps])

    idxs = selected_hidden_indices(hidden_states)
    layer_vecs = {}
    summary = []
    for group, hidx in idxs.items():
        v = hidden_states[hidx][0, -1, :].detach().float().cpu().numpy().astype(np.float32)
        layer_vecs[group] = v.astype(np.float16)
        summary.extend(summarize_vec(v))
    layer_vecs["all"] = np.concatenate([layer_vecs[g].astype(np.float16) for g in ["early", "middle", "late", "final"]]).astype(np.float16)

    row = {
        "example_id": example["id"],
        "dataset": example["dataset"],
        "n_options": len(labels),
        "answer_idx": int(example["answer_idx"]),
        "pred_idx": pred_idx,
        "correct": correct,
        "top_prob": top_prob,
        "margin": margin,
        "entropy": ent,
    }
    for i, val in enumerate(cheap):
        row[f"cheap_{i:02d}"] = float(val)
    for i, val in enumerate(summary):
        row[f"hsum_{i:02d}"] = float(val)

    return row, layer_vecs


def repeated_sampling_features(example, tokenizer, model, n_samples=5):
    prompt = format_mcq_prompt(example)
    prompt = apply_chat_template_if_available(tokenizer, prompt)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    labels = example["labels"]
    outputs = []
    with torch.no_grad():
        for _ in range(n_samples):
            gen = model.generate(
                **enc,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                max_new_tokens=3,
                pad_token_id=tokenizer.eos_token_id,
            )
            text = tokenizer.decode(gen[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip().upper()
            chosen = None
            for lab in labels:
                if text.startswith(lab):
                    chosen = lab
                    break
            outputs.append(chosen if chosen is not None else "UNK")
    counts = {lab: outputs.count(lab) for lab in labels}
    max_count = max(counts.values()) if counts else 0
    agreement = max_count / float(n_samples)
    unique = len(set(outputs))
    return {"sample_agreement": agreement, "sample_unique": unique}

## 5. Latency/VRAM benchmark

This benchmark is hardware-specific. It is included to support the cost-aware argument, not as a universal latency claim.

In [ ]:
def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def peak_vram_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.max_memory_reserved() / (1024 ** 3)


def benchmark_latency(model_short, model_id, tokenizer, model, n_prompts=80, warmup=5):
    out_path = TABLE_DIR / f"latency_{model_short}.csv"
    if out_path.exists() and not OVERWRITE:
        print("Latency exists, skipping:", out_path)
        return pd.read_csv(out_path)

    synthetic = []
    for i in range(n_prompts):
        ex = {
            "dataset": "synthetic",
            "id": f"syn_{i}",
            "question": f"A student mixes water and salt. Which option best describes the result? Example {i}.",
            "choices": ["A solution is formed", "A new animal is formed", "The Sun becomes colder", "Gravity disappears"],
            "labels": ["A", "B", "C", "D"],
            "answer_idx": 0,
        }
        synthetic.append(ex)

    def run_conf(ex, hidden=False):
        prompt = apply_chat_template_if_available(tokenizer, format_mcq_prompt(ex))
        enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            _ = model(**enc, output_hidden_states=hidden, use_cache=False)

    def run_gen5(ex):
        prompt = apply_chat_template_if_available(tokenizer, format_mcq_prompt(ex))
        enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            for _ in range(5):
                _ = model.generate(
                    **enc,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.95,
                    max_new_tokens=3,
                    pad_token_id=tokenizer.eos_token_id,
                )

    rows = []
    for signal_name, fn in [
        ("conf_option", lambda ex: run_conf(ex, hidden=False)),
        ("conf_plus_hidden_readout", lambda ex: run_conf(ex, hidden=True)),
        ("five_sample_generation", run_gen5),
    ]:
        # warmup
        for ex in synthetic[:warmup]:
            fn(ex)
        clear_gpu()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        times = []
        for ex in tqdm(synthetic, desc=f"Latency {model_short} {signal_name}"):
            cuda_sync()
            t0 = time.perf_counter()
            fn(ex)
            cuda_sync()
            times.append((time.perf_counter() - t0) * 1000.0)
        rows.append({
            "model": model_short,
            "model_id": model_id,
            "signal_family": signal_name,
            "mean_latency_ms": float(np.mean(times)),
            "p95_latency_ms": float(np.percentile(times, 95)),
            "peak_vram_gb": float(peak_vram_gb()),
        })
        clear_gpu()
    df = pd.DataFrame(rows)
    base = float(df.loc[df["signal_family"] == "conf_option", "mean_latency_ms"].iloc[0])
    df["relative_latency"] = df["mean_latency_ms"] / base
    df.to_csv(out_path, index=False)
    print(df)
    return df

## 6. Run feature extraction

This cell loads one model at a time, extracts features for all four datasets, saves each condition, then releases GPU memory.

In [ ]:
def feature_paths(model_short, dataset_name):
    prefix = FEATURE_DIR / f"{model_short}__{dataset_name}"
    return {
        "csv": prefix.with_suffix(".features.csv"),
        "npz": prefix.with_suffix(".hidden.npz"),
        "meta": prefix.with_suffix(".meta.json"),
    }


def save_condition_features(model_spec, dataset_name, tokenizer, model):
    model_short = model_spec["short_name"]
    paths = feature_paths(model_short, dataset_name)
    if paths["csv"].exists() and paths["npz"].exists() and not OVERWRITE:
        print(f"Features exist, skipping: {model_short} / {dataset_name}")
        return

    max_n = DEBUG_N if DEBUG_N is not None else MAX_EXAMPLES[dataset_name]
    rows = load_dataset_rows(dataset_name, max_n)
    print(f"Loaded {len(rows)} examples for {dataset_name}")

    feature_rows = []
    layer_store = {"early": [], "middle": [], "late": [], "final": [], "all": []}
    errors = []

    for ex in tqdm(rows, desc=f"Extract {model_short} {dataset_name}"):
        try:
            row, layer_vecs = score_one_example(ex, tokenizer, model)
            if RUN_REPEATED_SAMPLING_FEATURES:
                row.update(repeated_sampling_features(ex, tokenizer, model, n_samples=5))
            feature_rows.append(row)
            for k in layer_store:
                layer_store[k].append(layer_vecs[k])
        except Exception as e:
            errors.append({"id": ex.get("id", "unknown"), "error": repr(e)})
            if len(errors) <= 5:
                print("Example error:", errors[-1])
        if len(feature_rows) > 0 and len(feature_rows) % 50 == 0:
            clear_gpu()

    if not feature_rows:
        raise RuntimeError(f"No features extracted for {model_short}/{dataset_name}. Errors: {errors[:3]}")

    df = pd.DataFrame(feature_rows)
    df.to_csv(paths["csv"], index=False)

    npz_dict = {}
    for k, vals in layer_store.items():
        npz_dict[f"H_{k}"] = np.stack(vals).astype(np.float16)
    npz_dict["correct"] = df["correct"].values.astype(np.int64)
    np.savez_compressed(paths["npz"], **npz_dict)

    meta = {
        "model_short": model_short,
        "model_id": model_spec["model_id"],
        "dataset": dataset_name,
        "n_requested": max_n,
        "n_extracted": int(len(df)),
        "accuracy": float(df["correct"].mean()),
        "errors": errors[:20],
    }
    paths["meta"].write_text(json.dumps(meta, indent=2))
    if errors:
        pd.DataFrame(errors).to_csv(LOG_DIR / f"errors_{model_short}_{dataset_name}.csv", index=False)
    print("Saved:", paths["csv"], paths["npz"], "acc=", meta["accuracy"])


def run_feature_extraction_all():
    for spec in MODEL_SPECS:
        model_short = spec["short_name"]
        model_id = spec["model_id"]
        print("\n" + "=" * 90)
        print("Loading model:", model_id)
        print("=" * 90)
        clear_gpu()
        tokenizer, model = load_quantized_model_and_tokenizer(model_id)

        if RUN_LATENCY:
            try:
                benchmark_latency(model_short, model_id, tokenizer, model, n_prompts=20 if DEBUG_N is not None else 80)
            except Exception as e:
                print("Latency benchmark failed but continuing:", repr(e))

        for dataset_name in DATASET_NAMES:
            try:
                save_condition_features(spec, dataset_name, tokenizer, model)
            except Exception as e:
                print(f"FAILED condition {model_short}/{dataset_name}:", repr(e))
                with open(LOG_DIR / "condition_failures.txt", "a") as f:
                    f.write(f"{model_short}\t{dataset_name}\t{repr(e)}\n")
            clear_gpu()

        del model
        del tokenizer
        clear_gpu()

if RUN_FEATURE_EXTRACTION:
    run_feature_extraction_all()
else:
    print("RUN_FEATURE_EXTRACTION=False, skipping.")

## 7. Metrics, probes, and stronger contrastive hidden-state method

In [ ]:
def get_feature_columns(df, prefix):
    return [c for c in df.columns if c.startswith(prefix)]


def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi if i < n_bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)


def selective_metrics(y_correct, scores, coverages=(0.8, 0.6, 0.4), abstention_rate=0.2):
    y = np.asarray(y_correct).astype(int)
    s = np.asarray(scores).astype(float)
    n = len(y)
    order = np.argsort(-s)  # high score = answer
    y_sorted = y[order]
    failures_sorted = 1 - y_sorted
    ks = np.arange(1, n + 1)
    risks = np.cumsum(failures_sorted) / ks
    cov = ks / n
    aurc = float(np.trapz(risks, cov))

    # optimal ordering places correct examples first, then failures
    y_opt = np.sort(y)[::-1]
    risks_opt = np.cumsum(1 - y_opt) / ks
    aurc_opt = float(np.trapz(risks_opt, cov))
    eaurc = float(aurc - aurc_opt)

    out = {"AURC": aurc, "E_AURC": eaurc}
    for c in coverages:
        k = max(1, int(math.ceil(c * n)))
        retained = y_sorted[:k]
        risk = float(1.0 - retained.mean())
        out[f"risk_at_{int(c*100)}cov"] = risk
        out[f"acc_at_{int(c*100)}cov"] = float(retained.mean())

    # failure capture among abstained bottom fraction
    k_abs = max(1, int(math.ceil(abstention_rate * n)))
    abstained_idx = np.argsort(s)[:k_abs]
    total_fail = max(1, int((1 - y).sum()))
    captured = int((1 - y[abstained_idx]).sum())
    out[f"failure_capture_at_{int(abstention_rate*100)}abst"] = float(captured / total_fail)
    return out


def binary_metrics(y_true, y_score):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(y_score).astype(float)
    p = np.clip(p, 1e-7, 1 - 1e-7)
    out = {}
    if len(np.unique(y)) < 2:
        out["AUROC"] = np.nan
        out["AUPRC"] = np.nan
    else:
        out["AUROC"] = float(roc_auc_score(y, p))
        out["AUPRC"] = float(average_precision_score(y, p))
    out["Brier"] = float(brier_score_loss(y, p))
    out["ECE"] = expected_calibration_error(y, p)
    out.update(selective_metrics(y, p))
    return out


def safe_predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        z = model.decision_function(X)
        return 1 / (1 + np.exp(-z))
    raise ValueError("No probability method")


def make_classifiers(seed):
    return {
        "logreg": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs", random_state=seed),
        ),
        "mlp": make_pipeline(
            StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, max_iter=250, random_state=seed),
        ),
        "extratrees": ExtraTreesClassifier(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        ),
        "hgb": HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=150,
            l2_regularization=0.01,
            random_state=seed,
        ),
    }


def evaluate_best_classifier(X, y, train_idx, test_idx, seed, family_name):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    best = None
    for clf_name, clf in make_classifiers(seed).items():
        try:
            clf.fit(X_train, y_train)
            p = safe_predict_proba(clf, X_test)
            met = binary_metrics(y_test, p)
            row = {"family": family_name, "probe": clf_name, **met}
            if best is None or (np.nan_to_num(row["AUROC"], nan=-1) > np.nan_to_num(best["AUROC"], nan=-1)):
                best = row
        except Exception as e:
            continue
    if best is None:
        best = {"family": family_name, "probe": "failed", "AUROC": np.nan, "AUPRC": np.nan, "Brier": np.nan, "ECE": np.nan,
                "AURC": np.nan, "E_AURC": np.nan, "risk_at_80cov": np.nan, "risk_at_60cov": np.nan, "risk_at_40cov": np.nan,
                "acc_at_80cov": np.nan, "acc_at_60cov": np.nan, "acc_at_40cov": np.nan, "failure_capture_at_20abst": np.nan}
    return best


def fit_pca_train_test(X_train, X_test, dim, seed):
    dim_eff = int(min(dim, X_train.shape[0] - 2, X_train.shape[1]))
    dim_eff = max(2, dim_eff)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xtr_s = scaler.fit_transform(X_train.astype(np.float32))
    Xte_s = scaler.transform(X_test.astype(np.float32))
    pca = PCA(n_components=dim_eff, random_state=seed)
    Xtr_p = pca.fit_transform(Xtr_s)
    Xte_p = pca.transform(Xte_s)
    return Xtr_p, Xte_p, pca


class ContrastiveProbe(nn.Module):
    def __init__(self, in_dim, emb_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, emb_dim),
        )
        self.head = nn.Linear(emb_dim, 1)

    def forward(self, x):
        z = self.encoder(x)
        z_norm = F.normalize(z, dim=1)
        logit = self.head(z).squeeze(1)
        return z_norm, logit


def supervised_contrastive_loss(z, y, temperature=0.2):
    """Small binary supervised contrastive loss."""
    if z.shape[0] < 3:
        return torch.tensor(0.0, device=z.device)
    y = y.view(-1, 1)
    mask = torch.eq(y, y.T).float().to(z.device)
    logits = torch.matmul(z, z.T) / temperature
    logits = logits - torch.max(logits, dim=1, keepdim=True)[0].detach()
    logits_mask = torch.ones_like(mask) - torch.eye(mask.shape[0], device=z.device)
    mask = mask * logits_mask
    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)
    denom = mask.sum(1)
    valid = denom > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=z.device)
    mean_log_prob_pos = (mask * log_prob).sum(1)[valid] / denom[valid]
    return -mean_log_prob_pos.mean()


def train_contrastive_embeddings(X_train_raw, y_train, X_test_raw, seed, epochs=None):
    if epochs is None:
        epochs = CONTRASTIVE_EPOCHS
    # Pre-PCA for stable and fast contrastive training.
    Xtr_p, Xte_p, pca = fit_pca_train_test(X_train_raw, X_test_raw, CONTRASTIVE_PRE_PCA_DIM, seed)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr_p).astype(np.float32)
    Xte = scaler.transform(Xte_p).astype(np.float32)
    ytr = y_train.astype(np.float32)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ContrastiveProbe(Xtr.shape[1], emb_dim=CONTRASTIVE_EMB_DIM).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    X_tensor = torch.tensor(Xtr, dtype=torch.float32)
    y_tensor = torch.tensor(ytr, dtype=torch.float32)
    n = X_tensor.shape[0]
    rng = np.random.default_rng(seed)

    for ep in range(epochs):
        perm = rng.permutation(n)
        model.train()
        for start in range(0, n, CONTRASTIVE_BATCH_SIZE):
            idx = perm[start:start + CONTRASTIVE_BATCH_SIZE]
            xb = X_tensor[idx].to(device)
            yb = y_tensor[idx].to(device)
            z, logit = model(xb)
            bce = F.binary_cross_entropy_with_logits(logit, yb)
            scl = supervised_contrastive_loss(z, yb.long(), temperature=0.2)
            loss = bce + 0.20 * scl
            opt.zero_grad()
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        Xtr_t = torch.tensor(Xtr, dtype=torch.float32).to(device)
        Xte_t = torch.tensor(Xte, dtype=torch.float32).to(device)
        ztr, logit_tr = model(Xtr_t)
        zte, logit_te = model(Xte_t)
        pte = torch.sigmoid(logit_te).detach().cpu().numpy()
        ztr_np = ztr.detach().cpu().numpy()
        zte_np = zte.detach().cpu().numpy()
    del model
    clear_gpu()
    return ztr_np, zte_np, pte


def evaluate_contrastive(X_hidden, X_cheap, y, train_idx, test_idx, seed):
    Xh_train, Xh_test = X_hidden[train_idx], X_hidden[test_idx]
    Xc_train, Xc_test = X_cheap[train_idx], X_cheap[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    ztr, zte, p_hidden = train_contrastive_embeddings(Xh_train, y_train, Xh_test, seed)
    hidden_metrics = {"family": "contrastive_hidden", "probe": "supcon_mlp", **binary_metrics(y_test, p_hidden)}

    # Combine cheap features with learned contrastive representation, then use logistic regression.
    Xtr_comb = np.concatenate([Xc_train.astype(np.float32), ztr.astype(np.float32)], axis=1)
    Xte_comb = np.concatenate([Xc_test.astype(np.float32), zte.astype(np.float32)], axis=1)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
    clf.fit(Xtr_comb, y_train)
    p_comb = safe_predict_proba(clf, Xte_comb)
    comb_metrics = {"family": "cheap_plus_contrastive", "probe": "supcon_embedding_plus_logreg", **binary_metrics(y_test, p_comb)}
    return hidden_metrics, comb_metrics

## 8. Run analysis over saved features

In [ ]:
def load_condition_arrays(model_short, dataset_name):
    paths = feature_paths(model_short, dataset_name)
    if not paths["csv"].exists() or not paths["npz"].exists():
        return None
    df = pd.read_csv(paths["csv"])
    arr = np.load(paths["npz"])
    y = df["correct"].values.astype(int)
    cheap_cols = get_feature_columns(df, "cheap_")
    hsum_cols = get_feature_columns(df, "hsum_")
    X_cheap = df[cheap_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
    X_hsum = df[hsum_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
    H = {g: arr[f"H_{g}"].astype(np.float32) for g in ["early", "middle", "late", "final", "all"]}
    return df, y, X_cheap, X_hsum, H


def stratified_splits(y):
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2 or min(np.bincount(y)) < 2:
        return []
    sss = StratifiedShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_SEED)
    return list(sss.split(np.zeros(len(y)), y))


def run_family_analysis_condition(model_short, dataset_name):
    loaded = load_condition_arrays(model_short, dataset_name)
    if loaded is None:
        print("Missing features:", model_short, dataset_name)
        return []
    df, y, X_cheap, X_hsum, H = loaded
    splits = stratified_splits(y)
    if not splits:
        print("Not enough classes:", model_short, dataset_name, np.bincount(y))
        return []

    rows = []
    base_acc = float(y.mean())
    for split_id, (train_idx, test_idx) in enumerate(splits):
        seed = RANDOM_SEED + split_id
        y_train, y_test = y[train_idx], y[test_idx]

        # Cheap features
        res = evaluate_best_classifier(X_cheap, y, train_idx, test_idx, seed, "cheap_conf_option")
        rows.append({"model": model_short, "dataset": dataset_name, "split": split_id, "base_accuracy": base_acc, **res})

        # Hidden summary only
        res = evaluate_best_classifier(X_hsum, y, train_idx, test_idx, seed, "hidden_summaries")
        rows.append({"model": model_short, "dataset": dataset_name, "split": split_id, "base_accuracy": base_acc, **res})

        # Cheap + summary
        X_chsum = np.concatenate([X_cheap, X_hsum], axis=1)
        res = evaluate_best_classifier(X_chsum, y, train_idx, test_idx, seed, "cheap_plus_summaries")
        rows.append({"model": model_short, "dataset": dataset_name, "split": split_id, "base_accuracy": base_acc, **res})

        # PCA hidden and cheap + PCA hidden, using the all-layer group.
        Xh_train, Xh_test, _ = fit_pca_train_test(H["all"][train_idx], H["all"][test_idx], PCA_MAIN_DIM, seed)
        X_pca_full = np.zeros((len(y), Xh_train.shape[1]), dtype=np.float32)
        X_pca_full[train_idx] = Xh_train
        X_pca_full[test_idx] = Xh_test

        res = evaluate_best_classifier(X_pca_full, y, train_idx, test_idx, seed, "pca_hidden")
        rows.append({"model": model_short, "dataset": dataset_name, "split": split_id, "base_accuracy": base_acc, **res})

        X_cpca = np.concatenate([X_cheap, X_pca_full], axis=1)
        res = evaluate_best_classifier(X_cpca, y, train_idx, test_idx, seed, "cheap_plus_pca_hidden")
        rows.append({"model": model_short, "dataset": dataset_name, "split": split_id, "base_accuracy": base_acc, **res})

        # Stronger hidden-state method: supervised contrastive probe.
        try:
            hres, cres = evaluate_contrastive(H["all"], X_cheap, y, train_idx, test_idx, seed)
            rows.append({"model": model_short, "dataset": dataset_name, "split": split_id, "base_accuracy": base_acc, **hres})
            rows.append({"model": model_short, "dataset": dataset_name, "split": split_id, "base_accuracy": base_acc, **cres})
        except Exception as e:
            print("Contrastive failed:", model_short, dataset_name, split_id, repr(e))
            with open(LOG_DIR / "contrastive_failures.txt", "a") as f:
                f.write(f"{model_short}\t{dataset_name}\t{split_id}\t{repr(e)}\n")

    return rows


def run_pca_dim_sensitivity_condition(model_short, dataset_name):
    loaded = load_condition_arrays(model_short, dataset_name)
    if loaded is None:
        return []
    df, y, X_cheap, X_hsum, H = loaded
    splits = stratified_splits(y)
    rows = []
    for split_id, (train_idx, test_idx) in enumerate(splits):
        seed = RANDOM_SEED + split_id
        # cheap reference using logistic only for stable diagnostic
        cheap_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
        cheap_clf.fit(X_cheap[train_idx], y[train_idx])
        p_cheap = safe_predict_proba(cheap_clf, X_cheap[test_idx])
        cheap_auc = roc_auc_score(y[test_idx], p_cheap) if len(np.unique(y[test_idx])) > 1 else np.nan
        for dim in PCA_DIMS:
            try:
                Xtr_p, Xte_p, pca = fit_pca_train_test(H["all"][train_idx], H["all"][test_idx], dim, seed)
                Xtr_comb = np.concatenate([X_cheap[train_idx], Xtr_p], axis=1)
                Xte_comb = np.concatenate([X_cheap[test_idx], Xte_p], axis=1)
                clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
                clf.fit(Xtr_comb, y[train_idx])
                p = safe_predict_proba(clf, Xte_comb)
                auc = roc_auc_score(y[test_idx], p) if len(np.unique(y[test_idx])) > 1 else np.nan
                rows.append({
                    "model": model_short,
                    "dataset": dataset_name,
                    "split": split_id,
                    "pca_dim": dim,
                    "cheap_auroc": float(cheap_auc),
                    "cheap_plus_pca_auroc": float(auc),
                    "gain": float(auc - cheap_auc),
                    "pca_variance": float(np.sum(pca.explained_variance_ratio_)),
                })
            except Exception as e:
                continue
    return rows


def run_layer_localization_condition(model_short, dataset_name):
    loaded = load_condition_arrays(model_short, dataset_name)
    if loaded is None:
        return []
    df, y, X_cheap, X_hsum, H = loaded
    splits = stratified_splits(y)
    rows = []
    for split_id, (train_idx, test_idx) in enumerate(splits):
        seed = RANDOM_SEED + split_id
        cheap_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
        cheap_clf.fit(X_cheap[train_idx], y[train_idx])
        p_cheap = safe_predict_proba(cheap_clf, X_cheap[test_idx])
        cheap_auc = roc_auc_score(y[test_idx], p_cheap) if len(np.unique(y[test_idx])) > 1 else np.nan
        for group in ["early", "middle", "late", "final", "all"]:
            try:
                Xtr_p, Xte_p, pca = fit_pca_train_test(H[group][train_idx], H[group][test_idx], PCA_MAIN_DIM, seed)
                clf_h = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
                clf_h.fit(Xtr_p, y[train_idx])
                p_h = safe_predict_proba(clf_h, Xte_p)
                auc_h = roc_auc_score(y[test_idx], p_h) if len(np.unique(y[test_idx])) > 1 else np.nan

                Xtr_comb = np.concatenate([X_cheap[train_idx], Xtr_p], axis=1)
                Xte_comb = np.concatenate([X_cheap[test_idx], Xte_p], axis=1)
                clf_c = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
                clf_c.fit(Xtr_comb, y[train_idx])
                p_c = safe_predict_proba(clf_c, Xte_comb)
                auc_c = roc_auc_score(y[test_idx], p_c) if len(np.unique(y[test_idx])) > 1 else np.nan
                rows.append({
                    "model": model_short,
                    "dataset": dataset_name,
                    "split": split_id,
                    "layer_group": group,
                    "cheap_auroc": float(cheap_auc),
                    "hidden_only_auroc": float(auc_h),
                    "cheap_plus_layer_auroc": float(auc_c),
                    "gain_over_cheap": float(auc_c - cheap_auc),
                })
            except Exception:
                continue
    return rows


def run_group_redundancy_condition(model_short, dataset_name):
    loaded = load_condition_arrays(model_short, dataset_name)
    if loaded is None:
        return []
    df, y, X_cheap, X_hsum, H = loaded
    splits = stratified_splits(y)
    rows = []
    rng = np.random.default_rng(RANDOM_SEED)
    for split_id, (train_idx, test_idx) in enumerate(splits):
        seed = RANDOM_SEED + split_id
        try:
            Xtr_p, Xte_p, _ = fit_pca_train_test(H["all"][train_idx], H["all"][test_idx], PCA_MAIN_DIM, seed)
            Xtr_comb = np.concatenate([X_cheap[train_idx], Xtr_p], axis=1)
            Xte_comb = np.concatenate([X_cheap[test_idx], Xte_p], axis=1)
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
            clf.fit(Xtr_comb, y[train_idx])
            p = safe_predict_proba(clf, Xte_comb)
            base_auc = roc_auc_score(y[test_idx], p) if len(np.unique(y[test_idx])) > 1 else np.nan

            # Permute cheap block
            Xte_perm_cheap = Xte_comb.copy()
            perm = rng.permutation(Xte_perm_cheap.shape[0])
            Xte_perm_cheap[:, :X_cheap.shape[1]] = Xte_perm_cheap[perm, :X_cheap.shape[1]]
            p_pc = safe_predict_proba(clf, Xte_perm_cheap)
            auc_pc = roc_auc_score(y[test_idx], p_pc) if len(np.unique(y[test_idx])) > 1 else np.nan

            # Permute hidden block
            Xte_perm_hid = Xte_comb.copy()
            perm = rng.permutation(Xte_perm_hid.shape[0])
            Xte_perm_hid[:, X_cheap.shape[1]:] = Xte_perm_hid[perm, X_cheap.shape[1]:]
            p_ph = safe_predict_proba(clf, Xte_perm_hid)
            auc_ph = roc_auc_score(y[test_idx], p_ph) if len(np.unique(y[test_idx])) > 1 else np.nan

            rows.append({
                "model": model_short,
                "dataset": dataset_name,
                "split": split_id,
                "combined_auroc": float(base_auc),
                "auroc_after_permute_cheap": float(auc_pc),
                "auroc_after_permute_hidden": float(auc_ph),
                "drop_when_permute_cheap": float(base_auc - auc_pc),
                "drop_when_permute_hidden": float(base_auc - auc_ph),
            })
        except Exception:
            continue
    return rows


def run_all_analyses():
    family_rows = []
    pca_rows = []
    layer_rows = []
    red_rows = []
    base_rows = []

    for spec in MODEL_SPECS:
        model_short = spec["short_name"]
        for dataset_name in DATASET_NAMES:
            loaded = load_condition_arrays(model_short, dataset_name)
            if loaded is not None:
                df, y, X_cheap, X_hsum, H = loaded
                base_rows.append({
                    "model": model_short,
                    "model_id": spec["model_id"],
                    "dataset": dataset_name,
                    "n": int(len(y)),
                    "base_accuracy": float(np.mean(y)),
                    "failures": int((1 - y).sum()),
                })
            print("Analyzing", model_short, dataset_name)
            family_rows.extend(run_family_analysis_condition(model_short, dataset_name))
            pca_rows.extend(run_pca_dim_sensitivity_condition(model_short, dataset_name))
            layer_rows.extend(run_layer_localization_condition(model_short, dataset_name))
            red_rows.extend(run_group_redundancy_condition(model_short, dataset_name))

    base_df = pd.DataFrame(base_rows)
    family_df = pd.DataFrame(family_rows)
    pca_df = pd.DataFrame(pca_rows)
    layer_df = pd.DataFrame(layer_rows)
    red_df = pd.DataFrame(red_rows)

    base_df.to_csv(TABLE_DIR / "base_accuracy_gemma_llama.csv", index=False)
    family_df.to_csv(TABLE_DIR / "feature_family_metrics_gemma_llama.csv", index=False)
    pca_df.to_csv(TABLE_DIR / "pca_dimension_sensitivity_gemma_llama.csv", index=False)
    layer_df.to_csv(TABLE_DIR / "layer_localization_gemma_llama.csv", index=False)
    red_df.to_csv(TABLE_DIR / "group_redundancy_gemma_llama.csv", index=False)

    if len(family_df):
        summary = (family_df
                   .groupby(["family"], as_index=False)
                   .agg(AUROC_mean=("AUROC", "mean"), AUROC_std=("AUROC", "std"),
                        AURC_mean=("AURC", "mean"), EAURC_mean=("E_AURC", "mean"),
                        Risk80_mean=("risk_at_80cov", "mean"), Risk60_mean=("risk_at_60cov", "mean"),
                        Risk40_mean=("risk_at_40cov", "mean"),
                        FailureCapture20_mean=("failure_capture_at_20abst", "mean")))
        summary.to_csv(TABLE_DIR / "feature_family_summary_gemma_llama.csv", index=False)
        display(summary.sort_values("AUROC_mean", ascending=False))

    display(base_df)
    return base_df, family_df, pca_df, layer_df, red_df

if RUN_ANALYSIS:
    base_df, family_df, pca_df, layer_df, red_df = run_all_analyses()
else:
    print("RUN_ANALYSIS=False, skipping.")

## 9. Figures and final ZIP export

In [ ]:
import matplotlib.pyplot as plt


def save_fig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def make_figures():
    # Load tables if needed
    base_path = TABLE_DIR / "base_accuracy_gemma_llama.csv"
    fam_path = TABLE_DIR / "feature_family_metrics_gemma_llama.csv"
    pca_path = TABLE_DIR / "pca_dimension_sensitivity_gemma_llama.csv"
    layer_path = TABLE_DIR / "layer_localization_gemma_llama.csv"
    red_path = TABLE_DIR / "group_redundancy_gemma_llama.csv"
    lat_paths = list(TABLE_DIR.glob("latency_*.csv"))

    if base_path.exists():
        df = pd.read_csv(base_path)
        pivot = df.pivot(index="dataset", columns="model", values="base_accuracy")
        plt.figure(figsize=(7, 4))
        plt.imshow(pivot.values, aspect="auto")
        plt.colorbar(label="Base accuracy")
        plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=20, ha="right")
        plt.yticks(range(len(pivot.index)), pivot.index)
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                plt.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center")
        plt.title("Gemma/Llama base MCQ accuracy")
        save_fig(FIG_DIR / "fig_base_accuracy_heatmap.png")

    if fam_path.exists():
        fam = pd.read_csv(fam_path)
        summary = fam.groupby("family", as_index=False)["AUROC"].mean().sort_values("AUROC", ascending=False)
        plt.figure(figsize=(8, 4))
        plt.bar(summary["family"], summary["AUROC"])
        plt.xticks(rotation=30, ha="right")
        plt.ylabel("Mean AUROC")
        plt.title("Reliability prediction by feature family")
        save_fig(FIG_DIR / "fig_feature_family_auroc.png")

        sel_cols = ["risk_at_80cov", "risk_at_60cov", "risk_at_40cov", "AURC", "E_AURC", "failure_capture_at_20abst"]
        sel = fam.groupby("family", as_index=False)[sel_cols].mean()
        sel.to_csv(TABLE_DIR / "selective_prediction_summary_gemma_llama.csv", index=False)

    if pca_path.exists():
        pca = pd.read_csv(pca_path)
        g = pca.groupby("pca_dim", as_index=False).agg(
            cheap=("cheap_auroc", "mean"),
            cheap_pca=("cheap_plus_pca_auroc", "mean"),
            gain=("gain", "mean"),
        )
        plt.figure(figsize=(6, 4))
        plt.plot(g["pca_dim"], g["cheap"], marker="o", label="cheap")
        plt.plot(g["pca_dim"], g["cheap_pca"], marker="o", label="cheap + PCA hidden")
        plt.xscale("log", base=2)
        plt.xlabel("PCA dimension")
        plt.ylabel("Mean AUROC")
        plt.title("PCA dimension sensitivity")
        plt.legend()
        save_fig(FIG_DIR / "fig_pca_dimension_sensitivity.png")

        plt.figure(figsize=(6, 4))
        plt.axhline(0, linewidth=1)
        plt.plot(g["pca_dim"], g["gain"], marker="o")
        plt.xscale("log", base=2)
        plt.xlabel("PCA dimension")
        plt.ylabel("Mean AUROC gain")
        plt.title("Incremental value of PCA hidden embeddings")
        save_fig(FIG_DIR / "fig_pca_gain.png")

    if layer_path.exists():
        layer = pd.read_csv(layer_path)
        order = ["early", "middle", "late", "final", "all"]
        g = layer.groupby("layer_group", as_index=False).agg(
            hidden=("hidden_only_auroc", "mean"),
            gain=("gain_over_cheap", "mean"),
        )
        g["order"] = g["layer_group"].map({k: i for i, k in enumerate(order)})
        g = g.sort_values("order")
        plt.figure(figsize=(7, 4))
        plt.bar(g["layer_group"], g["hidden"])
        plt.ylabel("Hidden-only AUROC")
        plt.title("Layer localization of hidden reliability signal")
        save_fig(FIG_DIR / "fig_layer_hidden_only.png")

        plt.figure(figsize=(7, 4))
        plt.axhline(0, linewidth=1)
        plt.bar(g["layer_group"], g["gain"])
        plt.ylabel("Gain over cheap AUROC")
        plt.title("Layer incremental value over cheap signals")
        save_fig(FIG_DIR / "fig_layer_gain.png")

    if red_path.exists():
        red = pd.read_csv(red_path)
        vals = {
            "permute cheap": red["drop_when_permute_cheap"].mean(),
            "permute hidden": red["drop_when_permute_hidden"].mean(),
        }
        plt.figure(figsize=(5, 4))
        plt.bar(list(vals.keys()), list(vals.values()))
        plt.ylabel("Mean AUROC drop")
        plt.title("Group-permutation importance")
        save_fig(FIG_DIR / "fig_group_permutation.png")

    if lat_paths:
        lat = pd.concat([pd.read_csv(p) for p in lat_paths], ignore_index=True)
        lat.to_csv(TABLE_DIR / "latency_all_gemma_llama.csv", index=False)
        pivot = lat.pivot(index="model", columns="signal_family", values="relative_latency")
        plt.figure(figsize=(7, 4))
        x = np.arange(len(pivot.index))
        width = 0.25
        cols = list(pivot.columns)
        for j, col in enumerate(cols):
            plt.bar(x + (j - len(cols)/2) * width + width/2, pivot[col].values, width, label=col)
        plt.xticks(x, pivot.index, rotation=15, ha="right")
        plt.ylabel("Relative latency")
        plt.title("Measured latency overhead")
        plt.legend(fontsize=8)
        save_fig(FIG_DIR / "fig_latency_relative.png")


def zip_outputs():
    zip_path = Path("/kaggle/working/slm_gemma_llama_inline_outputs.zip") if Path("/kaggle").exists() else Path("./slm_gemma_llama_inline_outputs.zip")
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in OUT_DIR.rglob("*"):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(OUT_DIR.parent)))
    print("Created ZIP:", zip_path)
    return zip_path

make_figures()
zip_path = zip_outputs()
print("Done. Download this file from Kaggle:", zip_path)

## 10. How to run full mode after debug

If the debug run succeeds:

1. Go to **Session options → Restart session**.
2. Change `DEBUG_N = None` in the configuration cell.
3. Keep `OVERWRITE = False`.
4. Run all cells again.
5. Download `/kaggle/working/slm_gemma_llama_inline_outputs.zip` and upload it back to ChatGPT.

If Gemma or Llama still fails with a gated-repo error, the issue is Hugging Face access, not this notebook. Confirm that you accepted the model license while logged in to the same Hugging Face account used to create `HF_TOKEN`.